In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0,1,2,3")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("HF_HUB_ENABLE_XET", "0")

# ===== Imports =====
import sys
import re
import time
import requests
import random
import numpy as np
import pandas as pd
from datetime import datetime
from bs4 import BeautifulSoup
from bs4.element import Comment
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# ===== Configuration =====
PORT = 3000
MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"
TIMEOUT = 20.0
PER_GPU_MEM_GIB = 22
SEED = 42
SHUFFLE = True
NUM_EPISODES = 50
PRINT_EACH_STEP = True

LOG_DIR = "webshop_logs"
os.makedirs(LOG_DIR, exist_ok=True)
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
CSV_PATH = os.path.join(LOG_DIR, f"webshop_run_{TIMESTAMP}.csv")

# -----------------------------
# -----------------------------
WEBSHOP_URL = f"http://127.0.0.1:{PORT}"

use_bf16 = torch.cuda.is_available() and getattr(torch.cuda, "is_bf16_supported", lambda: False)()
dtype = torch.bfloat16 if use_bf16 else torch.float16

gpu_count = torch.cuda.device_count()
if gpu_count < 1 or not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU not found.")


random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
print(f"Random seed: {SEED}")

visible_gpus = min(gpu_count, 4)
max_memory = {i: f"{PER_GPU_MEM_GIB}GiB" for i in range(visible_gpus)}

# -----------------------------
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side="left", use_fast=True)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=dtype,
    device_map="auto",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id
model.eval()

def _model_primary_device(m):
    for p in m.parameters():
        return p.device
    return torch.device("cuda:0")

PRIMARY_DEVICE = _model_primary_device(model)
print(f"Primary device: {PRIMARY_DEVICE}")

# -----------------------------
# -----------------------------
def llm_with_metrics(prompt: str, stop=["\n"]) -> tuple:
    """
    
    Returns:
        (action_text, metrics_dict)
        metrics_dict: {
            'entropy': float,
            'top1_prob': float,
            'top2_prob': float,
            'top1_top2_margin': float,
            'top1_token': str,
            'top2_token': str,
            'perplexity': float
        }
    """
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False)
    inputs = {k: v.to(PRIMARY_DEVICE) for k, v in inputs.items()}
    
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
            use_cache=True,
            return_dict_in_generate=True,
            output_scores=True,  
        )
    
    gen_ids = out.sequences[:, inputs["input_ids"].shape[1]:]
    text = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)[0]
    
    if stop:
        cut = len(text)
        for s in stop:
            pos = text.find(s)
            if pos != -1:
                cut = min(cut, pos)
        text = text[:cut]
    
    metrics = compute_uncertainty_metrics(out.scores, gen_ids[0])
    
    return text.strip(), metrics

def compute_uncertainty_metrics(scores, generated_ids):
    
    if not scores or len(scores) == 0:
        return {
            'entropy': 0.0,
            'top1_prob': 1.0,
            'top2_prob': 0.0,
            'top1_top2_margin': 1.0,
            'top1_token': '',
            'top2_token': '',
            'perplexity': 1.0,
            'avg_entropy': 0.0,
            'avg_top1_prob': 1.0
        }
    
    first_logits = scores[0][0]  # (vocab_size,)
    first_probs = F.softmax(first_logits, dim=-1)
    
    # Entropy
    entropy = -torch.sum(first_probs * torch.log(first_probs + 1e-10)).item()
    
    # Top-k probabilities
    top_probs, top_indices = torch.topk(first_probs, k=min(5, len(first_probs)))
    top1_prob = top_probs[0].item()
    top2_prob = top_probs[1].item() if len(top_probs) > 1 else 0.0
    
    # Margin (confidence gap)
    margin = top1_prob - top2_prob
    
    # Top tokens
    top1_token = tokenizer.decode([top_indices[0].item()])
    top2_token = tokenizer.decode([top_indices[1].item()]) if len(top_indices) > 1 else ''
    
    all_entropies = []
    all_top1_probs = []
    log_probs = []
    
    for i, logits in enumerate(scores):
        probs = F.softmax(logits[0], dim=-1)
        ent = -torch.sum(probs * torch.log(probs + 1e-10)).item()
        all_entropies.append(ent)
        
        if i < len(generated_ids):
            selected_token_id = generated_ids[i].item()
            selected_prob = probs[selected_token_id].item()
            all_top1_probs.append(selected_prob)
            log_probs.append(np.log(selected_prob + 1e-10))
    
    avg_entropy = np.mean(all_entropies) if all_entropies else 0.0
    avg_top1_prob = np.mean(all_top1_probs) if all_top1_probs else 1.0
    
    perplexity = np.exp(-np.mean(log_probs)) if log_probs else 1.0
    
    return {
        'entropy': entropy,
        'top1_prob': top1_prob,
        'top2_prob': top2_prob,
        'top1_top2_margin': margin,
        'top1_token': top1_token.strip(),
        'top2_token': top2_token.strip(),
        'perplexity': perplexity,
        'avg_entropy': avg_entropy,
        'avg_top1_prob': avg_top1_prob,
        'num_tokens': len(scores)
    }

# -----------------------------
# -----------------------------
ACTION_TO_TEMPLATE = {
    'Description': 'description_page.html',
    'Features': 'features_page.html',
    'Reviews': 'review_page.html',
    'Attributes': 'attributes_page.html',
}

def clean_str(p):
    return p.encode().decode("unicode-escape").encode("latin1").decode("utf-8")

def tag_visible(element):
    ignore = {'style', 'script', 'head', 'title', 'meta', '[document]'}
    return element.parent.name not in ignore and not isinstance(element, Comment)

def webshop_text(session, page_type, query_string='', page_num=1, asin='', options={}, subpage='', **kwargs):
    if page_type == 'init':
        url = f'{WEBSHOP_URL}/{session}'
    if page_type == 'search':
        url = f'{WEBSHOP_URL}/search_results/{session}/{query_string}/{page_num}'
    elif page_type == 'item':
        url = f'{WEBSHOP_URL}/item_page/{session}/{asin}/{query_string}/{page_num}/{options}'
    elif page_type == 'item_sub':
        url = f'{WEBSHOP_URL}/item_sub_page/{session}/{asin}/{query_string}/{page_num}/{subpage}/{options}'
    elif page_type == 'end':
        url = f'{WEBSHOP_URL}/done/{session}/{asin}/{options}'

    html = requests.get(url, timeout=TIMEOUT).text
    html_obj = BeautifulSoup(html, 'html.parser')
    texts = html_obj.find_all(string=True)
    visible_texts = list(filter(tag_visible, texts))

    observation = ''
    option_type = ''
    options_map = {}
    asins = []
    cnt = 0
    prod_cnt = 0
    just_prod = 0
    for t in visible_texts:
        if t == '\n': 
            continue
        if t.replace('\n', '').replace('\\n', '').replace(' ', '') == '':
            continue
        if t.parent.name == 'button':
            processed_t = f'\n[{t}] '
        elif t.parent.name == 'label':
            if f"'{t}'" in url:
                processed_t = f'[[{t}]]'
            else:
                processed_t = f'[{t}]'
            options_map[str(t)] = option_type
        elif t.parent.get('class') == ["product-link"]:
            processed_t = f'\n[{t}] '
            if prod_cnt >= 3:
                processed_t = ''
            prod_cnt += 1
            asins.append(str(t))
            just_prod = 0
        else:
            processed_t = '\n' + str(t) + ' '
            if cnt < 2 and page_type != 'init':
                processed_t = ''
            if just_prod <= 2 and prod_cnt >= 4:
                processed_t = ''
            option_type = str(t)
            cnt += 1
        just_prod += 1
        observation += processed_t

    info = {}
    if options_map:
        info['option_types'] = options_map
    if asins:
        info['asins'] = asins
    if 'Your score (min 0.0, max 1.0)' in visible_texts:
        idx = visible_texts.index('Your score (min 0.0, max 1.0)')
        info['reward'] = float(visible_texts[idx + 1])
        observation = 'Your score (min 0.0, max 1.0): ' + (visible_texts[idx + 1])

    return clean_str(observation), info

class webshopEnv:
    def __init__(self):
        self.sessions = {}
    
    def step(self, session, action):
        done = False
        observation_ = None
        if action == 'reset':
            self.sessions[session] = {'session': session, 'page_type': 'init'}
        elif action.startswith('think['):
            observation = 'OK.'
        elif action.startswith('search['):
            assert self.sessions[session]['page_type'] == 'init'
            query = action[7:-1]
            self.sessions[session] = {'session': session, 'page_type': 'search',
                                      'query_string': query, 'page_num': 1}
        elif action.startswith('click['):
            button = action[6:-1]
            if button == 'Buy Now':
                assert self.sessions[session]['page_type'] == 'item'
                self.sessions[session]['page_type'] = 'end'
                done = True
            elif button == 'Back to Search':
                assert self.sessions[session]['page_type'] in ['search', 'item_sub', 'item']
                self.sessions[session] = {'session': session, 'page_type': 'init'}
            elif button == 'Next >':
                assert False
            elif button == '< Prev':
                assert self.sessions[session]['page_type'] in ['search', 'item_sub', 'item']
                if self.sessions[session]['page_type'] == 'search':
                    assert False
                elif self.sessions[session]['page_type'] == 'item_sub':
                    self.sessions[session]['page_type'] = 'item'
                elif self.sessions[session]['page_type'] == 'item':
                    self.sessions[session]['page_type'] = 'search'
                    self.sessions[session]['options'] = {}
            elif button in ACTION_TO_TEMPLATE:
                assert self.sessions[session]['page_type'] == 'item'
                self.sessions[session]['page_type'] = 'item_sub'
                self.sessions[session]['subpage'] = button
            else:
                if self.sessions[session]['page_type'] == 'search':
                    assert button in self.sessions[session].get('asins', [])
                    self.sessions[session]['page_type'] = 'item'
                    self.sessions[session]['asin'] = button
                elif self.sessions[session]['page_type'] == 'item':
                    assert 'option_types' in self.sessions[session]
                    assert button in self.sessions[session]['option_types'], (button, self.sessions[session]['option_types'])
                    option_type = self.sessions[session]['option_types'][button]
                    if 'options' not in self.sessions[session]:
                        self.sessions[session]['options'] = {}
                    self.sessions[session]['options'][option_type] = button
                    observation_ = f'You have clicked {button}.'
        else:
            assert False

        observation, info = webshop_text(**self.sessions[session])
        if observation_:
            observation = observation_
        self.sessions[session].update(info)
        reward = info.get('reward', 0.0)
        return observation, reward, done

env = webshopEnv()

# -----------------------------
# -----------------------------
prompt1 = """Webshop 
Instruction:  
i would like a 3 ounce bottle of bright citrus deodorant for sensitive skin, and price lower than 50.00 dollars 
[Search]  

Action: search[3 ounce bright citrus deodorant sensitive skin]
Observation: 
[Back to Search] 
Page 1 (Total results: 50) 
[Next >] 
[B078GWRC1J] 
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B078GTKVXY] 
Ginger Fresh Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
$10.99 
[B08KBVJ4XN] 
Barrel and Oak - Aluminum-Free Deodorant, Deodorant for Men, Essential Oil-Based Scent, 24-Hour Odor Protection, Cedar & Patchouli Blend, Gentle on Sensitive Skin (Mountain Sage, 2.7 oz, 2-Pack) 
$15.95  

Action: think[B078GWRC1J and B078GTKVXY are bright citrus deodorant less then 50 dollars. I can check B078GWRC1J first.]
Observation: OK.

Action: click[B078GWRC1J]
Observation: 
[Back to Search] 
[< Prev] 
scent [assorted scents][bright citrus][calming lavender][ginger fresh][simply non-scents]
size [travel set (4-pack)][3 ounce (pack of 1)][3-ounce (2-pack)]
Bright Citrus Deodorant by Earth Mama | Natural and Safe for Sensitive Skin, Pregnancy and Breastfeeding, Contains Organic Calendula 3-Ounce 
Price: $10.99 
Rating: N.A. 
[Description] 
[Features] 
[Reviews] 
[Buy Now]  

Action: think[For 3 ounce bottle of bright citrus deodorant for sensitive skin, the item has options 'bright citrus' and '3 ounce (pack of 1)' and seems good to buy.]
Observation: OK.

Action: click[bright citrus]
Observation: You have clicked bright citrus. 

Action: click[3 ounce (pack of 1)]
Observation: You have clicked 3 ounce (pack of 1). 

Action: click[Buy Now]
"""

# -----------------------------
# -----------------------------
class CSVLogger:
    def __init__(self, filepath):
        self.filepath = filepath
        self.logs = []
        
    def log_step(self, episode_id, step_num, action, observation, 
                 entropy, top1_prob, top2_prob, margin, top1_token, top2_token,
                 perplexity, avg_entropy, avg_top1_prob, num_tokens, reward, done):
        self.logs.append({
            'timestamp': datetime.now().isoformat(),
            'episode_id': episode_id,
            'step': step_num,
            'action': action,
            'observation': observation[:200],  
            'entropy': entropy,
            'top1_prob': top1_prob,
            'top2_prob': top2_prob,
            'top1_top2_margin': margin,
            'top1_token': top1_token,
            'top2_token': top2_token,
            'perplexity': perplexity,
            'avg_entropy': avg_entropy,
            'avg_top1_prob': avg_top1_prob,
            'num_tokens': num_tokens,
            'reward': reward,
            'done': done
        })
    
    def save(self):
        df = pd.DataFrame(self.logs)
        df.to_csv(self.filepath, index=False)
        return df

csv_logger = CSVLogger(CSV_PATH)

# -----------------------------
# -----------------------------
def webshop_run(idx, prompt, to_print=True, log_fn=None):
    if log_fn is None:
        log_fn = print

    action = 'reset'
    init_prompt = prompt
    prompt_buf = ''
    
    for step in range(15):
        try:
            observation, reward, done = env.step(idx, action)
        except AssertionError:
            observation, reward, done = 'Invalid action!', 0.0, False

        if action.startswith('think'):
            observation = 'OK.'

        if to_print:
            log_fn(f'Action: {action}\nObservation: {observation}\n')

        if step:
            prompt_buf += f' {action}\nObservation: {observation}\n\nAction:'
        else:
            prompt_buf = f'{observation}\n\nAction:'

        if done:
            return reward

        context_limit = 6400
        prefix = init_prompt
        suffix = prompt_buf[-(context_limit - len(prefix)):] if len(prefix) < context_limit else ''
        
        action, metrics = llm_with_metrics(prefix + suffix, stop=['\n'])
        action = action.lstrip(' ')
        
        if to_print:
            log_fn(f"[METRICS] Entropy={metrics['entropy']:.3f} | "
                   f"Top1={metrics['top1_prob']:.3f} | "
                   f"Margin={metrics['top1_top2_margin']:.3f} | "
                   f"Perplexity={metrics['perplexity']:.2f} | "
                   f"Top1Token='{metrics['top1_token']}' | "
                   f"Top2Token='{metrics['top2_token']}'")
        
        csv_logger.log_step(
            episode_id=idx,
            step_num=step + 1,
            action=action,
            observation=observation,
            entropy=metrics['entropy'],
            top1_prob=metrics['top1_prob'],
            top2_prob=metrics['top2_prob'],
            margin=metrics['top1_top2_margin'],
            top1_token=metrics['top1_token'],
            top2_token=metrics['top2_token'],
            perplexity=metrics['perplexity'],
            avg_entropy=metrics['avg_entropy'],
            avg_top1_prob=metrics['avg_top1_prob'],
            num_tokens=metrics['num_tokens'],
            reward=reward,
            done=done
        )

    return 0.0

def run_episodes(prompt, n=50, print_each_step=False):
    order = list(range(n))
    if SHUFFLE:
        rng = random.Random(SEED)
        rng.shuffle(order)

    rs = []
    failures = 0
    start_all = time.time()

    ep_bar = tqdm(order, desc="Running Episodes", unit="ep", dynamic_ncols=True)
    
    for rank, ep_idx in enumerate(ep_bar):
        t0 = time.time()
        try:
            r = webshop_run(
                f'fixed_{ep_idx}', 
                prompt,
                to_print=print_each_step,
                log_fn=tqdm.write
            )
        except Exception as e:
            if print_each_step:
                tqdm.write(f"Episode {ep_idx} error: {e}")
            r = 0.0
            failures += 1

        rs.append(r)
        avg_r = sum(rs) / len(rs)
        sr = sum(1 for _ in rs if _ == 1.0) / len(rs)
        fr = failures / len(rs)
        ep_time = time.time() - t0

        ep_bar.set_postfix({
            "R": f"{r:.2f}",
            "avg": f"{avg_r:.3f}",
            "succ%": f"{sr*100:.1f}",
            "fail%": f"{fr*100:.1f}",
            "ep_s": f"{ep_time:.1f}"
        })

    ep_bar.close()

    total_time = time.time() - start_all
    print(f"\n{'='*60}")
    print(f"{'='*60}")
    print(f"   - Avg Reward: {sum(rs)/len(rs):.4f}")
    print(f"   - Success Rate (R=1.0): {sum(1 for _ in rs if _ == 1.0)/len(rs)*100:.2f}%")
    print(f"   - Failure Rate (Error): {failures/len(rs)*100:.2f}%")
    print(f"{'='*60}")
    
    df = csv_logger.save()
    
    
    failed_actions = df[df['reward'] == 0.0]
    
    
    return rs

# -----------------------------
# -----------------------------

results = run_episodes(prompt1, n=NUM_EPISODES, print_each_step=PRINT_EACH_STEP)